In [ ]:
import asyncio
import json
from pathlib import Path

import httpx
import polars as pl

from experiments import nuextract_trial as nx
from experiments.extraction_common import load_requests
from experiments.harness import sample_items

PROGRESS = Path("../data/extraction/progress.jsonl")
TRIAL_OUT = Path("analysis_output/experiments/nuextract_trial")

pl.Config(tbl_cols=-1, tbl_width_chars=220, fmt_str_lengths=40)

In [ ]:
shards = pl.read_ndjson(PROGRESS)
display(
    shards.select(
        "shard", "model", "scale", "concurrency", "records", "requests",
        "resolved_ops", "errors", "prompt_tokens", "completion_tokens",
        "duration_s", "energy_wh",
        records_per_s=pl.col("records") / pl.col("duration_s"),
        wh_per_record=pl.col("energy_wh") / pl.col("records"),
        tokens_per_record=(pl.col("prompt_tokens") + pl.col("completion_tokens"))
        / pl.col("records"),
    )
)

done = shards.select(pl.col("records").sum(), pl.col("energy_wh").sum(),
                     pl.col("duration_s").sum()).row(0)
print(f"production so far: {done[0]:,} records, {done[1] / 1e3:.2f} kWh, {done[2] / 3600:.1f} h")
print(f"full queue at this rate: {nx.QUEUE_RECORDS * done[1] / done[0] / 1e3:.1f} kWh, "
      f"{nx.QUEUE_RECORDS * done[2] / done[0] / 3600:.0f} h")

In [ ]:
tasks = list(nx.TASK_KEYS)
print(nx.template_for(tasks), "\n")
print(nx.instructions_for(tasks))

In [ ]:
# One real request: document, raw reply, and the gate's verdict
items = sample_items("extraction", 3)
req = load_requests(items)[0]
doc = json.dumps(req["record_json"], indent=1, ensure_ascii=False)
print(doc[:1200], "…\n" if len(doc) > 1200 else "\n")

served = httpx.get("http://localhost:30000/v1/models", timeout=10.0).json()["data"][0]["id"]
print("served:", served)

In [ ]:
if served == "NuExtract3":
    from mds_norm.utils.inference import Inference

    group = nx._tasks_of(req)
    reply = asyncio.run(Inference(model=served, concurrency=1).generate(
        [doc], usage=True, progress=False, temperature=0.0, max_tokens=1024,
        chat_template_kwargs={"template": nx.template_for(group),
                              "instructions": nx.instructions_for(group),
                              "enable_thinking": False}))[0]
    print(reply["content"])
    print(f"\n{reply['prompt_tokens']} prompt + {reply['completion_tokens']} completion tokens")
    from mds_norm.utils.patches import validate_op

    from experiments.extraction_common import parse_ops_fields
    for op in parse_ops_fields(reply["content"]) or []:
        parsed, reason = validate_op(req, op)
        print(f"  {op['field']:<28} {op['value'][:40]!r:<44} "
              f"{'accepted' if parsed else 'REJECTED: ' + reason}")

In [ ]:
for name, variant in nx.VARIANTS.items():
    if variant.model != served:
        continue
    asyncio.run(nx.RUNNERS.get(name, nx.predict_nuextract)(variant, None))

In [ ]:
results = nx.score()
display(results.select("variant", "pred_ops", "precision_strict", "precision_judged",
                       "unjudged_ops", "recall_strict", "f1_strict", "placement_error",
                       "gold_ops_missed", "rejected", "deferred"))

In [ ]:
display(results.select("variant", "duration_s", "records_per_s", "prompt_tokens",
                       "completion_tokens", "tokens_per_accepted_op", "energy_wh",
                       "wh_per_record", "projected_queue_kwh"))

INCUMBENT = "gpt-oss-20b:single_task_v2"
base = results.filter(pl.col("variant") == INCUMBENT)
if base.height:
    b = base.row(0, named=True)
    print(f"incumbent here: {b['records_per_s']:.1f} rec/s, {b['wh_per_record']:.5f} Wh/record, "
          f"F1 {b['f1_strict']:.3f}, {b['prompt_tokens']:,} prompt tokens")
    for r in results.filter(pl.col("variant") != INCUMBENT).iter_rows(named=True):
        print(f"{r['variant']:<42} "
              f"{r['records_per_s'] / b['records_per_s']:>5.2f}x throughput  "
              f"{b['wh_per_record'] / r['wh_per_record']:>5.2f}x energy efficiency  "
              f"F1 {r['f1_strict']:.3f} ({r['f1_strict'] - b['f1_strict']:+.3f})")

In [ ]:
bench = (pl.read_ndjson(TRIAL_OUT / "bench.jsonl")
         .filter(pl.col("records") >= nx.BENCH_RECORDS)
         .unique(subset="variant", keep="last", maintain_order=True))
display(bench.select("variant", "records", "requests", "errors", "records_per_s",
                     "tokens_per_record", "wh_per_record", "projected_queue_kwh",
                     "projected_queue_h"))

# quality against cost, one line per variant: the frontier
frontier = (results.select("variant", "f1_strict", "recall_strict", "precision_strict")
            .join(bench.select("variant", "records_per_s", "wh_per_record",
                               "projected_queue_kwh", "projected_queue_h"),
                  on="variant", how="inner")
            .sort("projected_queue_kwh"))
display(frontier)